### **Import libraries**: 

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import sys
import fastparquet

### **Load Parquet file into a Pandas DataFrame object and calculate mean sentiment, sentiment standard deviation, number of articles, number of strongly negative articles, and sentiment momentum.**

In [2]:
def load_and_calculate(parquet_file_name_param, all_metadata):
    df = pd.read_parquet(parquet_file_name_param)
    tickers = list(set(df.index.get_level_values(0)))
    tickers.sort()
    dates = list(set(df.index.get_level_values(1)))
    dates = [str(element) for element in dates]
    dates.sort()
    date_today = str(datetime.today() - timedelta(days=2))[:10]
    iterables = [tickers, pd.to_datetime(dates)]
    index = pd.MultiIndex.from_product(iterables, names=["Ticker", "Timestamp"])
    df_aggregated = pd.DataFrame(index = index, columns = ["sent_mean", "sent_std", "news_count", "pct_strong_negative", "sent_momentum"])
    df_aggregated.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)

    counter = 0
    total = len(tickers) * len(dates)
    
    for ticker in tickers:
        for date in dates:
            try:
                sent_diff = df.loc[ticker, date]["pos"].sub(df.loc[ticker, date]["neg"], axis = 0)
                sent_mean_today = sent_diff.mean()
                sent_std_today = sent_diff.std()
                news_count_today = len(sent_diff)
                if news_count_today == 1:
                    sent_std_today = 0
                pct_strong_negative_today = (len(df.loc[ticker, date]["neg"][df.loc[ticker, date]["neg"]>0.7]) / len(df.loc[ticker, date]["neg"]))*100
                try:
                    sent_5_day_avg = np.mean([(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["pos"].sub(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["neg"], axis = 0)).mean() for i in range(5)])
                    sent_momentum = sent_mean_today - sent_5_day_avg
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, sent_momentum
                    
                except KeyError:
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, np.nan
                
            except KeyError:
                traceback.print_exc()
                sys.exit()
                #print("No article was published about " + str(ticker) + " today.")
        
            with open("status.txt", "w") as f:
                f.writelines(all_metadata)
                f.write(str(round((counter/total)*100, 2)) + "% complete." + "\n")

            counter +=1

    df_aggregated[df_aggregated.columns] = df_aggregated[df_aggregated.columns].apply(pd.to_numeric, axis = 1)
    return df_aggregated
    #df_aggregated.index.set_levels(new_time_index level=1, inplace=True)


### **Main program used to collectively execute the whole script.**

In [3]:
def main():

    with open("status.txt", "r+") as f:
        content = f.read()
        time_now = datetime.now()
        start_string = "Phase 3 started on " + time_now.strftime("%d-%m-%Y") + " at " + time_now.strftime("%H:%M:%S") + "." + "\n"
        f.write(start_string)
        f.seek(0)
        all_metadata = f.readlines()
    
    parquet_file_name = "5_year_ticker_headlines_with_finbert_rating.parquet"
    df_with_alt_data_features = load_and_calculate(parquet_file_name, all_metadata)
    
    df_with_alt_data_features.to_parquet("5_year_ticker_headlines_with_finbert_rating_calculated.parquet")

    with open("status.txt", "w") as f:
        f.writelines(all_metadata)
        time_now = datetime.now()
        end_string = "Phase 3 completed on " + time_now.strftime("%d-%m-%Y") + " at " + time_now.strftime("%H:%M:%S") + "." + "\n"
        f.write(end_string) 
        
main()

/tmp/ipykernel_2506/319375785.py:20: PerformanceWarning: indexing past lexsort depth may impact performance.
  sent_diff = df.loc[ticker, date]["pos"].sub(df.loc[ticker, date]["neg"], axis = 0)
Traceback (most recent call last):
  File "/home/leventefo/Programming/git/algorithmic-trading/notebooks/data_team/phase3/env/lib64/python3.14/site-packages/pandas/core/indexes/base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 197, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7668, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7676, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: '2021-04-23 08:00:00'

The above exception was the direct cause of the following exception:

Tracebac

SystemExit: 

/home/leventefo/Programming/git/algorithmic-trading/notebooks/data_team/phase3/env/lib64/python3.14/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
